# 루틴(routine) 플래너 평가 노트북

**대상:** "매주 독서하기" 같은 **반복 루틴**을 한 달(기본 28일) horizon 으로 펼치고,
사용자가 요일/주기를 **수정**하면 다시 펼치는 기능.

## 아키텍처 (뉴로-심볼릭)
```
사용자 입력 → judge_sufficiency(LLM)  : plan_kind=routine + slots{activity, cadence} 추출
            → plan_generator_node      : routine 이면 LLM 생략하고 코드로 전개
            → expand_routine(코드)      : cadence → horizon 내 요일별 캘린더 이벤트(결정적)
```

**평가 전략:** 전개(expand_routine)와 분기는 **결정적**이라 RunPod LLM 없이 재현 가능하게 평가한다.
LLM(judge) 의 슬롯 추출은 마지막 *라이브 API* 섹션에서 선택적으로 확인한다.

> 실행: 리포지토리 루트에서 `uv run jupyter lab`. 또는 비대화 실행:
> `uv run --with jupyter --with nbconvert jupyter nbconvert --to notebook --execute --inplace sft_pipeline/eval/routine_eval.ipynb`

## 0. 셋업

In [1]:
import sys
from datetime import date, timedelta
from pathlib import Path

# 리포 루트를 sys.path 에 추가(소스 패키지라 cwd 기반 import — 어디서 실행하든 동작).
_root = Path.cwd()
while _root != _root.parent and not (_root / 'agents').is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from agents.todo_creation.planner.allocator import expand_routine, _parse_weekdays

# 기준일 고정(재현성). 2026-05-27 = 수요일.
TODAY = date(2026, 5, 27)
_KO_WEEKDAY = ['월', '화', '수', '목', '금', '토', '일']
print('today:', TODAY, _KO_WEEKDAY[TODAY.weekday()] + '요일')

today: 2026-05-27 수요일


## 1. `expand_routine` — cadence → 날짜 전개 평가

여러 cadence 표현을 28일 horizon 으로 펼쳐 **요일 집합**과 **개수**를 확인한다.
참고: `_parse_weekdays` 규칙 — 명시 요일(월수금)이 있으면 그 요일, 없으면 `주 N회` 의 N 으로 분산,
`매주`처럼 숫자가 없으면 주 1회(월요일 기본).

In [2]:
def render_calendar(events, *, today=TODAY, weeks=4):
    """전개된 이벤트를 주 단위 달력 격자로 출력(■=이벤트 있는 날)."""
    marked = {e.due_date for e in events}
    start = today - timedelta(days=today.weekday())  # 그 주 월요일
    print('       월 화 수 목 금 토 일')
    for w in range(weeks):
        row = []
        for d in range(7):
            day = start + timedelta(days=w * 7 + d)
            if day < today:
                row.append(' . ')
            elif day in marked:
                row.append('[■]')
            else:
                row.append(' · ')
        wk = (start + timedelta(days=w * 7)).strftime('  %m/%d')
        print(wk, ''.join(row))


cadences = ['매주', '주3회', '월수금', '주7회', '화목']
for cad in cadences:
    events = expand_routine('독서', cad, today=TODAY, horizon_days=28)
    days = sorted({_KO_WEEKDAY[e.due_date.weekday()] for e in events})
    print(f"\ncadence={cad!r}  →  {len(events)}개 / 요일={days}")
    render_calendar(events)


cadence='매주'  →  4개 / 요일=['월']
       월 화 수 목 금 토 일
  05/25  .  .  ·  ·  ·  ·  · 
  06/01 [■] ·  ·  ·  ·  ·  · 
  06/08 [■] ·  ·  ·  ·  ·  · 
  06/15 [■] ·  ·  ·  ·  ·  · 

cadence='주3회'  →  12개 / 요일=['금', '수', '월']
       월 화 수 목 금 토 일
  05/25  .  . [■] · [■] ·  · 
  06/01 [■] · [■] · [■] ·  · 
  06/08 [■] · [■] · [■] ·  · 
  06/15 [■] · [■] · [■] ·  · 

cadence='월수금'  →  12개 / 요일=['금', '수', '월']
       월 화 수 목 금 토 일
  05/25  .  . [■] · [■] ·  · 
  06/01 [■] · [■] · [■] ·  · 
  06/08 [■] · [■] · [■] ·  · 
  06/15 [■] · [■] · [■] ·  · 

cadence='주7회'  →  28개 / 요일=['금', '목', '수', '월', '일', '토', '화']
       월 화 수 목 금 토 일
  05/25  .  . [■][■][■][■][■]
  06/01 [■][■][■][■][■][■][■]
  06/08 [■][■][■][■][■][■][■]
  06/15 [■][■][■][■][■][■][■]

cadence='화목'  →  8개 / 요일=['목', '화']
       월 화 수 목 금 토 일
  05/25  .  .  · [■] ·  ·  · 
  06/01  · [■] · [■] ·  ·  · 
  06/08  · [■] · [■] ·  ·  · 
  06/15  · [■] · [■] ·  ·  · 


### 1-a. 전개 규칙 검증 (assert)
각 cadence 가 의도한 요일 집합으로 전개되는지 단언한다.

In [3]:
EXPECTED = {
    '매주':   {0},            # 숫자 없음 → 주1회(월)
    '주3회':  {0, 2, 4},      # 분산
    '월수금': {0, 2, 4},      # 명시 요일
    '주7회':  {0, 1, 2, 3, 4, 5, 6},
    '화목':   {1, 3},
}
for cad, want in EXPECTED.items():
    ev = expand_routine('x', cad, today=TODAY, horizon_days=28)
    got = {e.due_date.weekday() for e in ev}
    assert got == want, f'{cad}: got {got} want {want}'
    assert all(TODAY <= e.due_date <= TODAY + timedelta(days=27) for e in ev)
print('✅ 전개 요일/horizon 규칙 통과')

✅ 전개 요일/horizon 규칙 통과


### 1-b. deadline clamp
마감일이 주어지면 그 이후 날짜는 만들지 않는다(Phase 0 규칙과 정합).

In [4]:
dl = TODAY + timedelta(days=10)
ev = expand_routine('독서', '주7회', today=TODAY, horizon_days=28, deadline=dl)
assert all(e.due_date <= dl for e in ev)
assert len(ev) == 11  # today..today+10 = 11일, 매일
print(f'deadline={dl} → {len(ev)}개, 모두 마감 이내 ✅')

deadline=2026-06-06 → 11개, 모두 마감 이내 ✅


## 2. `plan_generator_node` routine 분기

judge 가 채운 `parsed_goal`(plan_kind=routine + slots)을 받으면 **LLM 을 전혀 호출하지 않고**
전개하고, judge 의 `goal_tag` 로 태깅하며, today 인 건 todos·미래는 calendar_events 로 나눈다.

In [5]:
from dataclasses import dataclass

from agents.todo_creation.planner.nodes.plan_generator import plan_generator_node


@dataclass
class BoomLLM:
    """routine 경로가 LLM 을 부르면 즉시 실패시켜 '코드 전개'를 보증한다."""
    async def generate_plan(self, **_):
        raise AssertionError('routine 은 LLM(generate_plan) 을 호출하면 안 된다')
    async def generate_goal_tag(self, **_):
        raise AssertionError('routine 은 LLM(generate_goal_tag) 을 호출하면 안 된다')


async def run_node(parsed_goal):
    cfg = {'configurable': {'ports': type('P', (), {'llm': BoomLLM()})()}}
    return await plan_generator_node({'today': TODAY, 'parsed_goal': parsed_goal}, cfg)


goal = {
    'plan_kind': 'routine',
    'slots': {'activity': '독서', 'cadence': '월수금'},
    'goal_tag': '독서루틴',
    'deadline': None,
}
out = await run_node(goal)  # jupyter top-level await
print('summary:', out['summary_text'])
print('todos:', len(out['todos']), '| calendar_events:', len(out['calendar_events']))
print('tags 샘플:', out['calendar_events'][0].tags if out['calendar_events'] else None)
assert all(e.tags == ['독서루틴'] for e in out['calendar_events'])
assert all(e.due_date.weekday() in {0, 2, 4} for e in out['calendar_events'])
print('✅ routine 분기: LLM 미호출 + goal_tag 태깅 + 월수금 전개')

summary: '독서' 루틴을 월수금 기준으로 다음 28일 동안 잡아뒀어요.
todos: 1 | calendar_events: 11
tags 샘플: ['독서루틴']
✅ routine 분기: LLM 미호출 + goal_tag 태깅 + 월수금 전개


## 3. 날짜 수정 (revision) — 멀티턴

"매주 월요일 독서" → "**화요일로 바꿔줘**" 처럼 주기를 고치면, 같은 thread 에서 judge 가
cadence 슬롯을 갱신하고 **다시 전개**한다. (결정적 전개라 revision 텍스트를 직접 못 읽으므로,
routine 은 revision 시 judge 를 재실행하도록 배선되어 있다.)

아래는 **FakeLLM**(judge 응답을 시나리오로 주입) 으로 파이프라인 전체를 재현한다 — RunPod 불필요.

In [6]:
from dataclasses import field
from datetime import datetime

from agents.todo_creation.planner.pipeline import PlannerPorts, run
from agents.todo_creation.schemas import PlannerInput, SplitResult


@dataclass
class FakeJudgeLLM:
    """judge_sufficiency 응답만 시나리오로 주입(나머지 routine 은 코드가 처리)."""
    sufficiency: list = field(default_factory=list)
    async def judge_sufficiency(self, *, history, message, today, user_profile_memory=None):
        return self.sufficiency.pop(0)
    async def generate_follow_up_question(self, **_):
        return '언제/어떤 활동인가요?'
    async def generate_plan(self, **_):
        return ('', [])
    async def generate_goal_tag(self, *, parsed_goal, history):
        return str(parsed_goal.get('goal_tag') or '루틴')
    async def tag_plan(self, *, plan, parsed_goal):
        return plan
    async def split_tasks(self, *, prompt, today):
        return SplitResult(intent='plan', tasks=[])


def routine_goal(cadence):
    return {
        'intent': 'plan', 'plan_kind': 'routine',
        'slots': {'activity': '독서', 'cadence': cadence},
        'goal_tag': '독서', 'deadline': None,
    }


NOW = datetime(2026, 5, 27, 9, 0)
llm = FakeJudgeLLM(sufficiency=[(True, [], routine_goal('월')), (True, [], routine_goal('화'))])
ports = PlannerPorts(llm=llm)

first = await run(
    PlannerInput(user_id='u1', message='매주 월요일 독서하기', today=TODAY, thread_id=None),
    ports=ports, now=NOW)
first_days = sorted({_KO_WEEKDAY[e.due_date.weekday()] for e in first.calendar_events + first.todos})
print('1턴: 매주 월요일 독서  →  요일', first_days, '| 개수', len(first.calendar_events + first.todos))

second = await run(
    PlannerInput(user_id='u1', message='화요일로 바꿔줘', today=TODAY, thread_id=first.thread_id),
    ports=ports, now=NOW)
second_days = sorted({_KO_WEEKDAY[e.due_date.weekday()] for e in second.calendar_events + second.todos})
print('2턴: 화요일로 바꿔줘   →  요일', second_days, '| 개수', len(second.calendar_events + second.todos))

assert first_days == ['월'] and second_days == ['화']
print('✅ revision: 월 → 화 로 재전개됨')

1턴: 매주 월요일 독서  →  요일 ['월'] | 개수 4
2턴: 화요일로 바꿔줘   →  요일 ['화'] | 개수 4
✅ revision: 월 → 화 로 재전개됨


/Users/jpaper/Documents/projects/mong-studio/mongle-ai-phase1-revive/.venv/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 4. 평가 요약 (eval harness)

케이스별 기대치 대비 PASS/FAIL 을 한 표로 집계한다. CI/회귀의 사람용 요약.

In [7]:
cases = []

def check(name, cond):
    cases.append((name, bool(cond)))

for cad, want in EXPECTED.items():
    got = {e.due_date.weekday() for e in expand_routine('x', cad, today=TODAY, horizon_days=28)}
    check(f'cadence {cad} → {sorted(want)}', got == want)

check('매주 → 28일 내 월요일 4회', len(expand_routine('x', '매주', today=TODAY, horizon_days=28)) == 4)

_dl = TODAY + timedelta(days=6)
check('deadline clamp(매일,6일)=7개',
      len(expand_routine('x', '주7회', today=TODAY, horizon_days=28, deadline=_dl)) == 7)

_out = await run_node(routine_goal('월수금') | {'goal_tag': 'T'})
check('routine 분기 LLM 미호출 + goal_tag 태깅',
      all(e.tags == ['T'] for e in _out['calendar_events']))

check('revision 월→화 재전개', first_days == ['월'] and second_days == ['화'])

passed = sum(ok for _, ok in cases)
print(f'== 평가 결과: {passed}/{len(cases)} PASS ==\n')
for name, ok in cases:
    print(('✅' if ok else '❌'), name)
assert passed == len(cases), '일부 평가 실패'

== 평가 결과: 9/9 PASS ==

✅ cadence 매주 → [0]
✅ cadence 주3회 → [0, 2, 4]
✅ cadence 월수금 → [0, 2, 4]
✅ cadence 주7회 → [0, 1, 2, 3, 4, 5, 6]
✅ cadence 화목 → [1, 3]
✅ 매주 → 28일 내 월요일 4회
✅ deadline clamp(매일,6일)=7개
✅ routine 분기 LLM 미호출 + goal_tag 태깅
✅ revision 월→화 재전개


## 5. (선택) 라이브 API 로 judge 까지 평가

위 섹션은 judge 를 FakeLLM 으로 대체했다. **실제 슬롯 추출(LLM)** 까지 보려면 FastAPI 를 띄우고
비동기 잡(submit→poll)으로 `/v1/todo/chat` 을 호출한다. RunPod LLM 이 필요하다.

```bash
# 별도 터미널 (reload 끄기 — 인메모리 잡이 증발하지 않게)
uv run uvicorn api.main:app --port 8010
```

아래 셀은 서버가 떠 있을 때만 동작한다(기본은 실행하지 않도록 가드).

In [8]:
RUN_LIVE = False  # 서버(:8010)+RunPod 준비되면 True 로

if RUN_LIVE:
    import os
    import time

    import httpx

    base = 'http://localhost:8010'
    headers = {'X-API-Key': os.environ.get('MONGLE_API_KEY', '')}

    def chat(message, thread_id=None):
        body = {'mode': 'multi', 'user_id': 'eval', 'message': message, 'today': str(TODAY)}
        if thread_id:
            body['thread_id'] = thread_id
        r = httpx.post(f'{base}/v1/todo/chat', json=body, headers=headers, timeout=320)
        r.raise_for_status()
        job = (r.json().get('result') or {}).get('job_id')
        for _ in range(160):
            time.sleep(2)
            p = httpx.get(f'{base}/v1/todo/chat/{job}', headers=headers, timeout=30).json()
            if p.get('status') != 'pending':
                return p.get('result', p)
        raise TimeoutError('job 미완료')

    res = chat('매주 독서하기')
    print('kind:', res.get('kind'))
    for e in (res.get('calendar_events') or [])[:6]:
        print(' ', e.get('due_date'), e.get('title'))
else:
    print('RUN_LIVE=False — 라이브 평가는 건너뜀(서버 준비 후 True 로 설정)')

RUN_LIVE=False — 라이브 평가는 건너뜀(서버 준비 후 True 로 설정)
